# Gene ↔ Protein Relation-Wise Merge



## 0. Configuration

In [17]:
import os
import pandas as pd

BASE_DIR  = '/storage/Arushi/090526_EvoAge/kg_formation/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'
PROC_DIR  = BASE_DIR + 'processed_data/'
OUT_PATH  = BASE_DIR + 'processed_data_relation_wise_merge/generalised/GENE_PROTEIN/ALL_GENE_PROTEIN.parquet'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

In [18]:
# RecName dict (for sources that store UniProt ACs in head)
Uniprot_Recname = pd.read_csv(DB_DIR + 'uniprot/Uniprot_ID_Recname.csv')
Uniprot_Recname_dict = dict(zip(Uniprot_Recname['AC'], Uniprot_Recname['RecName']))

# Parsed TrEMBL dict (AC → NAME) for head-name fallback
uniprot = pd.read_csv(DB_DIR + 'uniprot/uniprot_parsed_results.csv')
uniprot_trembl_AC_Name_dict = dict(zip(uniprot['AC'], uniprot['NAME']))
del uniprot

print(f"UniProt RecName dict: {len(Uniprot_Recname_dict):,} | TrEMBL dict: {len(uniprot_trembl_AC_Name_dict):,}")

UniProt RecName dict: 794,151 | TrEMBL dict: 252,635,149


## 1. Load KG Sources

### 1a. CKG

In [6]:
CKG_Gene_Protein = pd.read_csv(PROC_DIR + 'CKG/File_26_Gene_protein_CKG.csv')
CKG_Gene_Protein.columns = CKG_Gene_Protein.columns.str.lower()
CKG_Gene_Protein['relation'] = 'Gene_Protein'
# Keep only rows with a head name
CKG_Gene_Protein = CKG_Gene_Protein[~CKG_Gene_Protein['head_detail_name'].isna()]
print(f"CKG:           {len(CKG_Gene_Protein):,} rows")

CKG_Gene_Protein['kg_type'] = 'Generalised'
CKG_Gene_Protein['kg_source'] = 'CKG'
CKG_Gene_Protein['species'] = 'Homo species'
CKG_Gene_Protein

CKG_Gene_Protein.head(2)

CKG:           579,366 rows


/tmp/ipykernel_2780776/4151918363.py:1: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  CKG_Gene_Protein = pd.read_csv(PROC_DIR + 'CKG/File_26_Gene_protein_CKG.csv')


,head,relation,tail,head_type,tail_type,source,kg_source,head_detail_name,head_ens,tail_alt_name,head_id_is,tail_id_is,kg_type,species
0,ADORA2A,Gene_Protein,P29274,Gene,Protein,UniProt,CKG,adenosine A2a receptor,ENSG00000128271,Adenosine receptor A2a,NCBI Gene Symbol,Uniprot_ID,Generalised,Homo species
1,HTR4,Gene_Protein,Q13639,Gene,Protein,UniProt,CKG,5-hydroxytryptamine receptor 4,ENSG00000164270,5-hydroxytryptamine receptor 4,NCBI Gene Symbol,Uniprot_ID,Generalised,Homo species


### 1b. Harmonizome 

In [9]:
Harmonizome_Gene_Protein_1 = pd.read_csv(PROC_DIR + 'harmonizome/Gene_Protein_Harmonizome.csv')
Harmonizome_Gene_Protein_1.columns = Harmonizome_Gene_Protein_1.columns.str.lower()
Harmonizome_Gene_Protein_1['relation'] = 'Gene_Protein'
# Keep only rows with a head name
print(f"harmonizome:{len(Harmonizome_Gene_Protein_1):,} rows")

Harmonizome_Gene_Protein_1['kg_type'] = 'Generalised'
Harmonizome_Gene_Protein_1['kg_source'] = 'Harmonizome'
Harmonizome_Gene_Protein_1['species'] = 'Homo species'
Harmonizome_Gene_Protein_1

Harmonizome_Gene_Protein_1.head(2)

harmonizome:286,985 rows


,head,relation,tail,head_type,tail_type,source,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,HNF4A,Gene_Protein,Q9UGM6,Gene,Protein,Pathway Commons Protein-Protein Interactions,Harmonizome,hepatocyte nuclear factor 4 alpha,"Tryptophan--tRNA ligase, mitochondrial",NCBI_ID,Uniprot_ID,Generalised,Homo species
1,UBC,Gene_Protein,Q9UGM6,Gene,Protein,Pathway Commons Protein-Protein Interactions,Harmonizome,ubiquitin C,"Tryptophan--tRNA ligase, mitochondrial",NCBI_ID,Uniprot_ID,Generalised,Homo species


## 2. Check and Fix Duplicate Columns

In [10]:
SOURCE_DFS = [
    ('CKG_Gene_Protein',           CKG_Gene_Protein),
    ('Harmonizome_Gene_Protein_1', Harmonizome_Gene_Protein_1),
   
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[CKG_Gene_Protein] ✓ no duplicates
[Harmonizome_Gene_Protein_1] ✓ no duplicates


## 3. Align Schemas and Concatenate

In [11]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name','kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

# Keep only rows with a head name
final_df = final_df[~final_df['head_detail_name'].isna()].reset_index(drop=True)
print(f"After head-name filter: {len(final_df):,} rows")

Combined: 866,351 rows
After head-name filter: 866,351 rows


## 4. Standardise Fixed-Value Columns

In [12]:
final_df['head_type']  = 'Gene'
final_df['tail_type']  = 'Protein'
final_df['relation']   = 'Gene_Protein'
final_df['head_id_is'] = 'NCBI_ID'

print("Unique relation:",   set(final_df['relation']))
print("Unique tail_id_is:", set(final_df['tail_id_is']))
print("Unique kg_source:",  set(final_df['kg_source']))

Unique relation: {'Gene_Protein'}
Unique tail_id_is: {'Uniprot_ID'}
Unique kg_source: {'CKG', 'Harmonizome'}


In [14]:
cols = [
    'relation', 'head_type', 'relation_type',
    'tail_type', 'kg_source', 'head_id_is',
    'tail_id_is', 'kg_type', 'species'
]

unique_vals = {
    col: set(final_df[col].dropna())
    for col in cols
}

unique_vals

{'relation': {'Gene_Protein'},
 'head_type': {'Gene'},
 'relation_type': set(),
 'tail_type': {'Protein'},
 'kg_source': {'CKG', 'Harmonizome'},
 'head_id_is': {'NCBI_ID'},
 'tail_id_is': {'Uniprot_ID'},
 'kg_type': {'Generalised'},
 'species': {'Homo species'}}

## 5. Deduplicate and Add Schema Columns

In [15]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
        'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

final_df_dedup = final_df_dedup[REQUIRED_COLS]
final_df_dedup.head()

After deduplication: 479,965 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,A1BG,Gene_Protein,A0ABJ7H345,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,None,Generalised,Homo species
1,A1BG,Gene_Protein,A0ABJ7H347,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,None,Generalised,Homo species
2,A1BG,Gene_Protein,A0ABJ7H8K2,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,None,Generalised,Homo species
3,A1BG,Gene_Protein,M0R009,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,None,Generalised,Homo species
4,A1BG,Gene_Protein,O43866,Gene,None,Protein,Harmonizome,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,CD5 antigen-like,Generalised,Homo species


In [27]:
#

## 6. QC — NaN Check

In [16]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,479965,0,479965
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


In [19]:
final_df_dedup[final_df_dedup['tail_detail_name'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,A1BG,Gene_Protein,A0ABJ7H345,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,None,Generalised,Homo species
1,A1BG,Gene_Protein,A0ABJ7H347,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,None,Generalised,Homo species
2,A1BG,Gene_Protein,A0ABJ7H8K2,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,None,Generalised,Homo species
3,A1BG,Gene_Protein,M0R009,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,None,Generalised,Homo species
8,A1BG,Gene_Protein,P04217,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,None,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
479958,ZZZ3,Gene_Protein,Q8IYH5,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,zinc finger ZZ-type containing 3,None,Generalised,Homo species
479959,ZZZ3,Gene_Protein,Q8IYH5-1,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,zinc finger ZZ-type containing 3,None,Generalised,Homo species
479960,ZZZ3,Gene_Protein,Q8IYH5-2,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,zinc finger ZZ-type containing 3,None,Generalised,Homo species
479961,ZZZ3,Gene_Protein,Q8IYH5-3,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,zinc finger ZZ-type containing 3,None,Generalised,Homo species


In [24]:
final_df_dedup['tail'] = final_df_dedup['tail'].str.replace(r'-.*$', '', regex=True)
final_df_dedup['tail_detail_name'] = (
    final_df_dedup['tail_detail_name']
    .fillna(final_df_dedup['tail'].map(uniprot_trembl_AC_Name_dict))
    .fillna(final_df_dedup['tail'].map(Uniprot_Recname_dict))
)
final_df_dedup

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,A1BG,Gene_Protein,A0ABJ7H345,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,NaN,Generalised,Homo species
1,A1BG,Gene_Protein,A0ABJ7H347,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,NaN,Generalised,Homo species
2,A1BG,Gene_Protein,A0ABJ7H8K2,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,NaN,Generalised,Homo species
3,A1BG,Gene_Protein,M0R009,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,Alternative protein MYO18B {ECO:0000313|EMBL:C...,Generalised,Homo species
4,A1BG,Gene_Protein,O43866,Gene,None,Protein,Harmonizome,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,CD5 antigen-like,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
479960,ZZZ3,Gene_Protein,Q8IYH5,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,zinc finger ZZ-type containing 3,ZZ-type zinc finger-containing protein 3,Generalised,Homo species
479961,ZZZ3,Gene_Protein,Q8IYH5,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,zinc finger ZZ-type containing 3,ZZ-type zinc finger-containing protein 3,Generalised,Homo species
479962,ZZZ3,Gene_Protein,Q8IYH5,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,zinc finger ZZ-type containing 3,ZZ-type zinc finger-containing protein 3,Generalised,Homo species
479963,ZZZ3,Gene_Protein,Q9NZI6,Gene,None,Protein,Harmonizome,NCBI_ID,Uniprot_ID,zinc finger ZZ-type containing 3,Transcription factor CP2-like protein 1,Generalised,Homo species


In [26]:
final_df_dedup[final_df_dedup['tail_detail_name'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,A1BG,Gene_Protein,A0ABJ7H345,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,NaN,Generalised,Homo species
1,A1BG,Gene_Protein,A0ABJ7H347,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,NaN,Generalised,Homo species
2,A1BG,Gene_Protein,A0ABJ7H8K2,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,NaN,Generalised,Homo species
973,ABCC1,Gene_Protein,A0A0G2JNC3,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,ATP binding cassette subfamily C member 1 (ABC...,NaN,Generalised,Homo species
4095,ACOX1,Gene_Protein,A0ABF7PH99,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,acyl-CoA oxidase 1,NaN,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
478399,ZNF717,Gene_Protein,A0ABJ7H8Q2,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,zinc finger protein 717,NaN,Generalised,Homo species
478825,ZNF79,Gene_Protein,A0ABJ7H339,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,zinc finger protein 79,NaN,Generalised,Homo species
478826,ZNF79,Gene_Protein,A0ABJ7HDM7,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,zinc finger protein 79,NaN,Generalised,Homo species
479024,ZNF840P,Gene_Protein,A0ABB0MV31,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,"zinc finger protein 840, pseudogene",NaN,Generalised,Homo species


In [28]:
final_df_dedup = final_df_dedup[~final_df_dedup['tail_detail_name'].isna()]
final_df_dedup

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
3,A1BG,Gene_Protein,M0R009,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,Alternative protein MYO18B {ECO:0000313|EMBL:C...,Generalised,Homo species
4,A1BG,Gene_Protein,O43866,Gene,None,Protein,Harmonizome,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,CD5 antigen-like,Generalised,Homo species
5,A1BG,Gene_Protein,P01859,Gene,None,Protein,Harmonizome,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,Immunoglobulin heavy constant gamma 2 {ECO:000...,Generalised,Homo species
6,A1BG,Gene_Protein,P02743,Gene,None,Protein,Harmonizome,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,Serum amyloid P-component(1-203),Generalised,Homo species
7,A1BG,Gene_Protein,P04196,Gene,None,Protein,Harmonizome,NCBI_ID,Uniprot_ID,alpha-1-B glycoprotein,Histidine-rich glycoprotein,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
479960,ZZZ3,Gene_Protein,Q8IYH5,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,zinc finger ZZ-type containing 3,ZZ-type zinc finger-containing protein 3,Generalised,Homo species
479961,ZZZ3,Gene_Protein,Q8IYH5,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,zinc finger ZZ-type containing 3,ZZ-type zinc finger-containing protein 3,Generalised,Homo species
479962,ZZZ3,Gene_Protein,Q8IYH5,Gene,None,Protein,CKG,NCBI_ID,Uniprot_ID,zinc finger ZZ-type containing 3,ZZ-type zinc finger-containing protein 3,Generalised,Homo species
479963,ZZZ3,Gene_Protein,Q9NZI6,Gene,None,Protein,Harmonizome,NCBI_ID,Uniprot_ID,zinc finger ZZ-type containing 3,Transcription factor CP2-like protein 1,Generalised,Homo species


## 7. Save Output

In [29]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_parquet(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 479,667 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/GENE_PROTEIN/ALL_GENE_PROTEIN.parquet
